# Baseline V1 — brief client → recommandation

Moteur de reference pour la Gate 1, a battre par la V2.

1. **Division** (Flavors / Fragrance) : comptage de mots-cles dans le brief.
2. **Recherche** : TF-IDF du brief contre `product_name` + `ingredients_text` + `query_category`,
   restreinte a la division detectee, similarite cosinus, top 5 avec score.

Donnees : `data/processed/sample_v1.csv` (2 193 produits, France, langue `fr`).
Aucun apprentissage : pas de donnees d'entrainement etiquetees a ce stade.

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)

DATA = Path("..") / "data" / "processed" / "sample_v1.csv"
catalogue = pd.read_csv(DATA, dtype=str)
catalogue.groupby(["division", "query_category"]).size()

division   query_category        
flavors    boissons_aromatisees      200
           boissons_lactees          200
           boissons_proteinees       200
           boissons_vegetales        200
           desserts_aromatises       200
fragrance  deodorants                315
           gels_douche               400
           parfums                    88
           savons                    315
           savons_rasage               3
           shampoings_gels_douche     72
dtype: int64

## 1. Detection de la division

Normalisation commune (minuscules, sans accents), puis comptage des mots-cles de chaque division
(match en debut de mot : `proteine` couvre `proteinee`). La division qui a le plus de mots-cles gagne ;
en cas d'egalite, `indeterminee`.

In [2]:
def normaliser(texte: str) -> str:
    texte = unicodedata.normalize("NFKD", str(texte).lower())
    return "".join(ch for ch in texte if not unicodedata.combining(ch))


MOTS_CLES = {
    "flavors": [
        "arome", "aromatise", "gout", "saveur", "boisson", "proteine", "yaourt", "lait",
        "dessert", "sirop", "soda", "the glace", "cafe", "smoothie", "creme dessert",
        "vanille", "fraise", "chocolat", "menthe", "caramel", "sportif",
    ],
    "fragrance": [
        "parfum", "senteur", "odeur", "fragrance", "savon", "deodorant", "gel douche",
        "douche", "shampoing", "cosmetique", "eau de toilette", "boise", "floral", "musc",
        "ambre", "frais", "fraiche", "hygiene", "peau",
    ],
}
# mots a matcher en entier : "frais" ne doit pas compter dans "fraise"
MOTS_ENTIERS = {"frais"}
MOTIFS = {
    div: [re.compile(r"\b" + re.escape(mot) + (r"\b" if mot in MOTS_ENTIERS else ""))
          for mot in mots]
    for div, mots in MOTS_CLES.items()
}


def detecter_division(brief: str) -> tuple[str, dict]:
    texte = normaliser(brief)
    scores = {div: sum(bool(m.search(texte)) for m in motifs) for div, motifs in MOTIFS.items()}
    if scores["flavors"] == scores["fragrance"]:
        return "indeterminee", scores
    return max(scores, key=scores.get), scores

## 2. Recherche TF-IDF

Texte indexe : `product_name` + `ingredients_text` + `query_category` (underscores remplaces par des
espaces). Unigrammes et bigrammes, accents retires, TF sous-lineaire. Un index par division pour
que l'IDF reflete le vocabulaire propre a chaque division.

In [3]:
STOP_FR = [
    "a", "au", "aux", "avec", "ce", "de", "des", "du", "en", "et", "la", "le", "les", "pour",
    "sur", "un", "une", "par", "base", "gamme", "goût", "gout",
]


def texte_produit(df: pd.DataFrame) -> pd.Series:
    return (
        df["product_name"].fillna("") + " "
        + df["ingredients_text"].fillna("") + " "
        + df["query_category"].fillna("").str.replace("_", " ")
    ).map(normaliser)


INDEX = {}
for division, df_div in catalogue.groupby("division"):
    vectoriseur = TfidfVectorizer(
        ngram_range=(1, 2), sublinear_tf=True, strip_accents="unicode",
        stop_words=[normaliser(m) for m in STOP_FR],
    )
    matrice = vectoriseur.fit_transform(texte_produit(df_div))
    INDEX[division] = (df_div.reset_index(drop=True), vectoriseur, matrice)
    print(f"{division}: {matrice.shape[0]} produits, {matrice.shape[1]} termes")

flavors: 1000 produits, 17151 termes
fragrance: 1193 produits, 18492 termes


In [4]:
COLONNES = ["code", "product_name", "brands", "query_category", "format", "score"]


def recommander(brief: str, k: int = 5) -> dict:
    division, scores_div = detecter_division(brief)
    # division indeterminee : on cherche dans les deux index et on garde les meilleurs scores
    divisions = list(INDEX) if division == "indeterminee" else [division]
    resultats = []
    for div in divisions:
        df_div, vectoriseur, matrice = INDEX[div]
        sims = cosine_similarity(vectoriseur.transform([normaliser(brief)]), matrice).ravel()
        top = df_div.iloc[sims.argsort()[::-1][:k]].copy()
        top["score"] = sims[sims.argsort()[::-1][:k]].round(3)
        resultats.append(top)
    top5 = pd.concat(resultats).sort_values("score", ascending=False).head(k)
    return {"division": division, "scores_division": scores_div, "top5": top5[COLONNES]}

## 3. Test sur 5 briefs

Division attendue fixee a la main pour chaque brief.

In [5]:
BRIEFS = [
    ("boisson protéinée vanille pour sportifs, base pois", "flavors"),
    ("arôme fraise pour yaourt à boire", "flavors"),
    ("senteur premium fraîche pour une gamme de savons", "fragrance"),
    ("déodorant parfum boisé pour homme", "fragrance"),
    ("sirop goût menthe pour cafés", "flavors"),
]

bilan = []
for brief, attendue in BRIEFS:
    res = recommander(brief)
    correcte = res["division"] == attendue
    bilan.append({"brief": brief, "attendue": attendue, "detectee": res["division"],
                  "mots_cles": res["scores_division"], "correcte": correcte,
                  "score_top1": res["top5"]["score"].iloc[0]})
    print(f"\n=== {brief}")
    print(f"division detectee : {res['division']} {res['scores_division']} "
          f"-> attendue {attendue} : {'CORRECTE' if correcte else 'INCORRECTE'}")
    display(res["top5"])


=== boisson protéinée vanille pour sportifs, base pois
division detectee : flavors {'flavors': 4, 'fragrance': 0} -> attendue flavors : CORRECTE


,code,product_name,brands,query_category,format,score
755,3257985312964,Boisson au soja vanille,Nature Bio,boissons_vegetales,NaN,0.103
745,10901809,Boisson Soja Vanille,"Bio Village, Marque Repère",boissons_vegetales,NaN,0.093
687,3229820806006,Barista avoine vanille,Bjorg,boissons_vegetales,NaN,0.090
751,3021760406403,Dessert Végétal Vanille de Madagascar,Materne,boissons_vegetales,NaN,0.090
583,5400706614726,Protéine De Pois Et Riz en Poudre,Purasana,boissons_proteinees,poudre,0.089



=== arôme fraise pour yaourt à boire
division detectee : flavors {'flavors': 3, 'fragrance': 0} -> attendue flavors : CORRECTE


,code,product_name,brands,query_category,format,score
374,3335030030404,Yaourt à boire fermier Fraise,le Craulois,boissons_lactees,NaN,0.213
301,3596710434619,Yaourt a boiresaveur fraise,Auchan,boissons_lactees,NaN,0.196
259,3258561420189,Yaourt à boire fraise,Belle France,boissons_lactees,NaN,0.191
256,3329770063273,Yaourt à boire sucré aromatisé,YoplaitYop,boissons_lactees,NaN,0.189
245,3350033646797,Yaourt à boire fraise,Monoprix,boissons_lactees,NaN,0.186



=== senteur premium fraîche pour une gamme de savons
division detectee : fragrance {'flavors': 0, 'fragrance': 3} -> attendue fragrance : CORRECTE


,code,product_name,brands,query_category,format,score
881,5410036269626,Senteur thé vert,"Dettol, Reckitt Benckiser",savons,NaN,0.175
672,3700214851029,Senteur Amande,"Gopack, Sans marque",savons,NaN,0.170
250,3298839232083,Eau de senteur pour bébé,Christine Arbel,parfums,NaN,0.154
253,3331840871633,Eau de toilette senteur Vanille Framboise,Les petits plaisirs,parfums,NaN,0.125
1053,8712561112765,Monsavon Déodorant Femme Spray Grenade Tellement Fraîche...,Monsavon,deodorants,NaN,0.106



=== déodorant parfum boisé pour homme
division detectee : fragrance {'flavors': 0, 'fragrance': 3} -> attendue fragrance : CORRECTE


,code,product_name,brands,query_category,format,score
414,3560070917488,Déodorant parfum floral,Carrefour,deodorants,NaN,0.294
416,3560070923465,déodorant parfum musk 24h simply choice,"Carrefour, simply choice",deodorants,NaN,0.217
1064,8712561456234,Déodorant Homme Spray Apollo,Axe,deodorants,NaN,0.120
990,8710847885563,AXE Ice Cool Déodorant Homme Anti Transpirant Homme Ment...,Axe,deodorants,NaN,0.110
977,8710522559383,AXE Déodorant Homme Spray Wild Lot 6X150ml,AXE,deodorants,NaN,0.103



=== sirop goût menthe pour cafés
division detectee : flavors {'flavors': 4, 'fragrance': 0} -> attendue flavors : CORRECTE


,code,product_name,brands,query_category,format,score
111,3660668209510,Sirop de Menthe,Frigolet,boissons_aromatisees,NaN,0.420
40,3092712200226,Sirop de Menthe Verte,Teisseire,boissons_aromatisees,NaN,0.399
42,3560070339228,Sirop Menthe,Simpl,boissons_aromatisees,NaN,0.319
34,5400113504276,Sirop De Menthe Bio,Delhaize,boissons_aromatisees,NaN,0.255
83,3350033184169,"Sirop à diluer à la menthe, sans sucre",Monoprix,boissons_aromatisees,NaN,0.170


In [6]:
bilan_df = pd.DataFrame(bilan)
print(f"Division correcte : {bilan_df['correcte'].sum()} / {len(bilan_df)}")
bilan_df

Division correcte : 5 / 5


,brief,attendue,detectee,mots_cles,correcte,score_top1
0,"boisson protéinée vanille pour sportifs, base pois",flavors,flavors,"{'flavors': 4, 'fragrance': 0}",True,0.103
1,arôme fraise pour yaourt à boire,flavors,flavors,"{'flavors': 3, 'fragrance': 0}",True,0.213
2,senteur premium fraîche pour une gamme de savons,fragrance,fragrance,"{'flavors': 0, 'fragrance': 3}",True,0.175
3,déodorant parfum boisé pour homme,fragrance,fragrance,"{'flavors': 0, 'fragrance': 3}",True,0.294
4,sirop goût menthe pour cafés,flavors,flavors,"{'flavors': 4, 'fragrance': 0}",True,0.420


## Lecture des resultats et limites de la baseline

- La detection de division repose sur une liste de mots-cles ecrite a la main : un brief sans aucun
  mot de la liste tombe en `indeterminee`, et un mot ambigu (`frais`, `menthe`, `vanille` existent
  dans les deux divisions) peut faire basculer la decision.
- Le TF-IDF compare des mots, pas des sens : `boise` ne trouve pas `cedre` ou `santal`,
  `sportifs` ne trouve pas `whey`, `premium` n'a pas d'equivalent dans les fiches produits.
- `ingredients_text` est long et domine le vecteur : le nom du produit et la categorie pesent peu.
- Les scores ne sont pas comparables d'une division a l'autre (index separes).
- Pas de verite terrain produit : seule la division est evaluee ici. La validation sur 30 cas de
  reference (jeudi) devra noter aussi la pertinence du top 5.

Pistes V2 : enrichir les mots-cles par le vocabulaire olfactif/aromatique (IFRA, familles de notes),
ponderer les champs, embeddings de phrases multilingues, extraction d'application et de notes.